# Gold Product Identifiers & Field-Level Lineage

Creates stable `GoldProductId` values, builds field-level provenance for the nine survivorship attributes, and publishes `gold.products` and `gold.productlineage`.

**Prerequisites:** Run the Entity Resolution & Product Grouping notebook and Product Survivorship Engine notebook. The survivorship output must be persisted as `gold.consolidatedproducts`; the group map must be persisted as `semantic.productgroupmembers`.

In [1]:
# Cell 1 – Load inputs
from pyspark.sql import functions as F

ConsolidatedProductsDf = spark.table("gold.consolidatedproducts")
GroupMembersDf = spark.table("semantic.productgroupmembers")

print("Consolidated groups:", ConsolidatedProductsDf.count())
print("Group members:", GroupMembersDf.count())

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 3, Finished, Available, Finished, False)

Consolidated groups: 142
Group members: 164


In [2]:
# Cell 2 – Create deterministic GoldProductId from sorted source-qualified group members
GroupSignaturesDf = (
    GroupMembersDf
    .groupBy("GoldProductGroupId")
    .agg(
        F.array_sort(
            F.collect_set(
                F.concat("SourceSystem", F.lit(":"), "SourceProductId")
            )
        ).alias("GroupMembers")
    )
    .withColumn("GroupSignature", F.concat_ws("|", "GroupMembers"))
    .withColumn(
        "GoldProductId",
        F.concat(F.lit("GP-"), F.sha2("GroupSignature", 256))
    )
    .select("GoldProductGroupId", "GoldProductId", "GroupSignature")
)

display(GroupSignaturesDf)

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b6d06f0-12bd-49d2-97fb-275a2fa73458)

In [6]:
# Cell 3 – Attach GoldProductId to survivorship output
GoldProductsWithIdDf = (
    ConsolidatedProductsDf
    .drop("GoldProductId")
    .join(
        GroupSignaturesDf.select("GoldProductGroupId", "GoldProductId"),
        "GoldProductGroupId",
        "inner"
    )
)

display(GoldProductsWithIdDf.limit(20))

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 462a877a-7c97-448f-bed8-5b70ea1b4702)

## Field-level lineage

Each selected attribute becomes one row in `GoldProductLineageDf`. `SelectedValue` is stored as a string for a consistent lineage schema; the typed selected value remains in `gold.products`.

In [9]:
LineageAttributes = [
    "ProductName", "Description", "Price", "Brand", "Category",
    "Color", "Size", "Material", "InStock"
]

RequiredColumns = [
    col
    for attr in LineageAttributes
    for col in [
        attr,
        f"{attr}SourceSystem",
        f"{attr}SourceProductID",
        f"{attr}SourceUpdatedAt",
        f"{attr}SurvivorshipRule"
    ]
]

MissingColumns = set(RequiredColumns) - set(GoldProductsWithIdDf.columns)

if MissingColumns:
    raise ValueError(
        f"Missing survivorship metadata columns: {sorted(MissingColumns)}"
    )

print("All required survivorship columns are available.")

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 11, Finished, Available, Finished, False)

All required survivorship columns are available.


In [10]:
# Cell 5 – Build normalized field-level lineage
LineageDfs = []

for AttributeName in LineageAttributes:
    LineageDfs.append(
        GoldProductsWithIdDf.select(
            "GoldProductId",
            F.lit(AttributeName).alias("FieldName"),
            F.col(AttributeName).cast("string").alias("SelectedValue"),
            F.col(f"{AttributeName}SourceSystem").alias("SourceSystem"),
            F.col(f"{AttributeName}SourceProductId").alias("SourceRecordId"),
            F.col(f"{AttributeName}SourceUpdatedAt").cast("string").alias("SourceUpdatedAt"),
            F.col(f"{AttributeName}SurvivorshipRule").alias("SurvivorshipRule")
        )
    )

GoldProductLineageDf = LineageDfs[0]
for LineageDf in LineageDfs[1:]:
    GoldProductLineageDf = GoldProductLineageDf.unionByName(LineageDf)

display(GoldProductLineageDf.orderBy("GoldProductId", "FieldName"))

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3157a23d-feb2-4576-a24a-0b2771a73468)

In [11]:
# Cell 6 – Prepare final Gold product schema (remove per-field helper metadata)
HelperColumns = [
    col
    for attr in LineageAttributes
    for col in [
        f"{attr}SourceSystem",
        f"{attr}SourceProductId",
        f"{attr}SourceUpdatedAt",
        f"{attr}SurvivorshipRule"
    ]
]

GoldProductsDf = GoldProductsWithIdDf.drop(*HelperColumns, "GroupSignature")
display(GoldProductsDf.limit(20))

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0228ee34-e741-417d-bd32-47d5dc4f853e)

## Publish final Gold Delta tables

This notebook uses overwrite for reproducible rebuilds. It does not preserve historical snapshots.

In [14]:
# Cell 7 – Write final tables
GoldProductsDf.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.products")

GoldProductLineageDf.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.productlineage")

print("Published gold.products and gold.productlineage")

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 16, Finished, Available, Finished, False)

Published gold.products and gold.productlineage


In [15]:
# Cell 8 – Read back published tables
GoldProductsReadDf = spark.table("gold.products")
GoldLineageReadDf = spark.table("gold.productlineage")

print("Gold product rows:", GoldProductsReadDf.count())
print("Gold lineage rows:", GoldLineageReadDf.count())
display(GoldProductsReadDf.limit(10))
display(GoldLineageReadDf.limit(20))

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 17, Finished, Available, Finished, False)

Gold product rows: 142
Gold lineage rows: 1278


SynapseWidget(Synapse.DataFrame, 45076f0e-db52-4e63-b6ef-87890ec1080f)

SynapseWidget(Synapse.DataFrame, 51b80a21-e8b3-407b-b90b-7ad27559ed1a)

In [16]:
# Cell 9 – Structural validation
DuplicateGoldIdsDf = (
    GoldProductsReadDf.groupBy("GoldProductId")
    .count().filter(F.col("count") > 1)
)

DuplicateLineageDf = (
    GoldLineageReadDf.groupBy("GoldProductId", "FieldName")
    .count().filter(F.col("count") > 1)
)

NonNullFieldsDf = GoldProductsReadDf.select(
    "GoldProductId",
    F.explode(
        F.array(*[
            F.when(F.col(attr).isNotNull(), F.lit(attr))
            for attr in LineageAttributes
        ])
    ).alias("FieldName")
).filter(F.col("FieldName").isNotNull())

MissingLineageDf = NonNullFieldsDf.join(
    GoldLineageReadDf.select("GoldProductId", "FieldName"),
    ["GoldProductId", "FieldName"],
    "left_anti"
)

print("Duplicate GoldProductId:", DuplicateGoldIdsDf.count())
print("Duplicate product-field lineage:", DuplicateLineageDf.count())
print("Non-null fields missing lineage:", MissingLineageDf.count())

if DuplicateGoldIdsDf.limit(1).count():
    display(DuplicateGoldIdsDf)
    raise ValueError("GoldProductId uniqueness check failed.")

if DuplicateLineageDf.limit(1).count():
    display(DuplicateLineageDf)
    raise ValueError("Duplicate lineage rows found.")

if MissingLineageDf.limit(1).count():
    display(MissingLineageDf)
    raise ValueError("Some non-null product fields have no lineage.")

print("Gold identifier and lineage structural checks passed.")

StatementMeta(, 8591262a-afe7-44c6-8d23-1579473d3fc0, 18, Finished, Available, Finished, False)

Duplicate GoldProductId: 0
Duplicate product-field lineage: 0
Non-null fields missing lineage: 0
Gold identifier and lineage structural checks passed.
